### DATA 611 - Group Project 2
By Group 6:
- Alvin Hui Tsz Chuen
- Binh Nguyen
- Judy Kurupakorn

### Data Wrangling and EDA

In [ ]:
# Import libraries
import pandas as pd
import numpy as np

In [ ]:
# Load datasets
vehicle_insurance_df = pd.read_csv('motor_data14-2018.csv')

# Select relevant columns and rename them for clarity
selected_columns = {
    "SEX": "sex",
    "INSR_BEGIN": "insurance_begin",
    "INSR_END": "insurance_end",
    "EFFECTIVE_YR": "effective_year",
    "INSR_TYPE": "insurance_type",
    "INSURED_VALUE": "insured_value",
    "PREMIUM": "premium",
    "OBJECT_ID": "object_id",
    "PROD_YEAR": "production_year",
    "SEATS_NUM": "seats_number",
    "CARRYING_CAPACITY": "carrying_capacity",
    "TYPE_VEHICLE": "vehicle_type",
    "CCM_TON": "ccm_ton",
    "MAKE": "manufacturer",
    "USAGE": "usage",
    "CLAIM_PAID": "claim_paid",
}

vehicle_insurance_df = vehicle_insurance_df[list(selected_columns.keys())].rename(columns=selected_columns)

# Replace sex values
vehicle_insurance_df["sex"] = vehicle_insurance_df["sex"].replace({0: "legal entity", 1: "male", 2: "female"})

# Replace insurance_type values
vehicle_insurance_df["insurance_type"] = vehicle_insurance_df["insurance_type"].replace({1201: "private", 1202: "commercial", 1204: "motor trade road risk"})

# Display data info
print(vehicle_insurance_df.info())
print(vehicle_insurance_df.head())

In [ ]:
# Drop duplicates
vehicle_insurance_df.drop_duplicates(inplace=True)

# Check for missing values
missing_values = vehicle_insurance_df.isnull().sum()
print("Missing values in each column:")
print(missing_values)

# Drop null values for premium, production_year, and manufacturer
vehicle_insurance_df.dropna(subset=["premium", "production_year", "manufacturer"], inplace=True)

# Convert production_year to an integer type
vehicle_insurance_df["production_year"] = vehicle_insurance_df["production_year"].astype(int)

# Check for missing values after dropping
missing_values_after = vehicle_insurance_df.isnull().sum()
print("Missing values after dropping:")
print(missing_values_after)
vehicle_insurance_df.dropna(subset=["premium", "production_year", "manufacturer"], inplace=True)

# Convert categorical variables to proper types
vehicle_insurance_df["sex"] = vehicle_insurance_df["sex"].astype("category")
vehicle_insurance_df["insurance_type"] = vehicle_insurance_df["insurance_type"].astype("category")
vehicle_insurance_df["vehicle_type"] = vehicle_insurance_df["vehicle_type"].astype("category")
vehicle_insurance_df["manufacturer"] = vehicle_insurance_df["manufacturer"].astype("category")
vehicle_insurance_df["usage"] = vehicle_insurance_df["usage"].astype("category")

# Check for zero values in each selected column
zero_values = (vehicle_insurance_df == 0).sum()
print("Zero values in each column:")
print(zero_values)

# Drop zero values in the premium column
vehicle_insurance_df = vehicle_insurance_df[vehicle_insurance_df["premium"] > 0]
vehicle_insurance_df.dropna(subset=["premium", "production_year", "manufacturer"], inplace=True)

# Replace 0s with median in the insured value column
median_insured_value_df = vehicle_insurance_df[vehicle_insurance_df['insured_value'] > 0].groupby(['manufacturer', 'vehicle_type', 'production_year'],
                                                                                                  dropna=False)['insured_value'].median().dropna().reset_index()
median_insured_value_dict = median_insured_value_df.set_index(['manufacturer', 'vehicle_type', 'production_year'])['insured_value'].to_dict()

def find_closest_year_match(manufacturer, vehicle_type, production_year):
    """Find the closest production year match if an exact match is not available, within a maximum difference of 5 years."""
    subset = median_insured_value_df[(median_insured_value_df['manufacturer'] == manufacturer) &
                                     (median_insured_value_df['vehicle_type'] == vehicle_type)]
    if subset.empty:
        return np.nan
    subset = subset[(subset['production_year'] - production_year).abs() <= 5]
    if subset.empty:
        return np.nan
    closest_year = subset.loc[(subset['production_year'] - production_year).abs().idxmin()]
    return closest_year['insured_value'] if not closest_year.empty else np.nan

def replace_zero_with_median(row):
    """Replace zero insured_value with the median value based on relaxed conditions."""
    key = (row['manufacturer'], row['vehicle_type'], row['production_year'])
    median_value = median_insured_value_dict.get(key, np.nan)

    if pd.notna(median_value) and median_value > 0:
        return median_value
    else:
        return find_closest_year_match(row['manufacturer'], row['vehicle_type'], row['production_year'])

# Apply replacements efficiently using vectorized approach
mask = vehicle_insurance_df['insured_value'] == 0
vehicle_insurance_df.loc[mask, 'insured_value'] = vehicle_insurance_df.loc[mask].apply(replace_zero_with_median, axis=1)

# Check for zero values after replacing
zero_values = (vehicle_insurance_df == 0).sum()
print("Zero values in each column after replacing:")
print(zero_values)

# Drop remaining zero-value rows in insured_value
vehicle_insurance_df = vehicle_insurance_df[vehicle_insurance_df['insured_value'] > 0]

# Check for zero values after dropping
zero_values = (vehicle_insurance_df == 0).sum()
print("Zero values in each column after dropping:")
print(zero_values)

In [ ]:
# Convert insurance_end and insurance_begin to date format
vehicle_insurance_df["insurance_begin"] = pd.to_datetime(vehicle_insurance_df["insurance_begin"], errors='coerce')
vehicle_insurance_df["insurance_end"] = pd.to_datetime(vehicle_insurance_df["insurance_end"], errors='coerce')

# Calculate insurance_duration in days
vehicle_insurance_df["insurance_duration"] = (vehicle_insurance_df["insurance_end"] - vehicle_insurance_df["insurance_begin"]).dt.days

# Create insurance_claim column
vehicle_insurance_df["insurance_claim"] = vehicle_insurance_df["claim_paid"].apply(lambda x: "claimed" if pd.notna(x) else "no claim")

# Display cleaned data info
print(vehicle_insurance_df.info())
print(vehicle_insurance_df.head())

# Save the updated DataFrame (optional)
# vehicle_insurance_df.to_csv("/mnt/data/updated_vehicle_insurance.csv", index=False)

In [ ]:
vehicle_insurance_df[['insured_value', 'premium', 'production_year', 'claim_paid']].describe().apply(lambda x: x.map('{:,.0f}'.format))

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
######## pie chart gender
sex_counts = vehicle_insurance_df['sex'].value_counts()
fig = go.Figure(data=[go.Pie(labels=sex_counts.index, values=sex_counts.values)])
fig.update_layout(
    title="Distribution of Sex",
    height=400,
    width=500,
    showlegend=True
)
fig.show()

######## pie chart insurance_type
insurance_type_counts = vehicle_insurance_df['insurance_type'].value_counts()
fig = go.Figure(data=[go.Pie(labels=insurance_type_counts.index, values=insurance_type_counts.values)])
fig.update_layout(
    title="Distribution of Insurance Type",
    height=400,
    width=500,
    showlegend=True
)
fig.show()

######## bar chart vehicle_type
vehicle_type_counts = vehicle_insurance_df['vehicle_type'].value_counts().sort_values(ascending=True)
fig = go.Figure(data=[go.Bar(x=vehicle_type_counts.values, y=vehicle_type_counts.index,orientation='h',marker=dict(color='darkgreen'))])
fig.update_layout(
    title="Distribution of Vehicle Type",
    height=500,
    width=700,
    plot_bgcolor='rgba(0,0,0,0)',
    showlegend=False
)
fig.show()

######## bar chart usage
usage_counts = vehicle_insurance_df['usage'].value_counts().sort_values(ascending=True)
fig = go.Figure(data=[go.Bar(x=usage_counts.values, y=usage_counts.index,orientation='h',marker=dict(color='darkgreen'))])
fig.update_layout(
    title="Distribution of Usage",
    height=500,
    width=700,
    plot_bgcolor='rgba(0,0,0,0)',
    showlegend=False
)

fig.show()


In [ ]:
# Check the distribution of production_year, insured_value, and premium
num_cols = ["production_year", "insured_value", "premium"]
figs = []
for col in num_cols:
    fig = px.histogram(vehicle_insurance_df, x=col, nbins=50, marginal="box",
                       opacity=0.9, color_discrete_sequence=['darkcyan'])
    fig.update_traces(marker_line_color='black', marker_line_width=0.5)
    fig.update_layout(
        height=400,
        width=550,
        plot_bgcolor='rgba(0,0,0,0)',
        showlegend=False
    )
    figs.append(fig)

for fig in figs:
    fig.show()

In [ ]:
# Take log of insured_value and premium to check distributions
vehicle_insurance_df["log_insured_value"] = vehicle_insurance_df["insured_value"].apply(lambda x: np.log(x) if x > 0 else 0)
vehicle_insurance_df["log_premium"] = vehicle_insurance_df["premium"].apply(lambda x: np.log(x) if x > 0 else 0)

num_cols = ["log_insured_value", "log_premium"]
figs = []
for col in num_cols:
    fig = px.histogram(vehicle_insurance_df, x=col, nbins=50, marginal="box",
                       color_discrete_sequence=['darkcyan'])
    fig.update_traces(marker_line_color='black', marker_line_width=0.5)
    fig.update_layout(
        height=400,
        width=550,
        plot_bgcolor='rgba(0,0,0,0)',
        showlegend=False
    )
    figs.append(fig)

for fig in figs:
    fig.show()

In [ ]:
# Plot average premium vs production year
average_premium_per_year = vehicle_insurance_df.groupby('production_year')['premium'].mean()
average_premium_private = vehicle_insurance_df[vehicle_insurance_df['insurance_type'] == 'private'].groupby('production_year')['premium'].mean()
average_premium_commercial = vehicle_insurance_df[vehicle_insurance_df['insurance_type'] == 'commercial'].groupby('production_year')['premium'].mean()

#average_premium_per_year
#average_premium_private
#average_premium_commercial

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=average_premium_per_year.index,
    y=average_premium_per_year.values,
    mode='lines',
    line=dict(color='darkblue'),
    name='Overall Average Premium'
))
fig.add_trace(go.Scatter(
    x=average_premium_private.index,
    y=average_premium_private.values,
    mode='lines',
    line=dict(color='red', dash='dash'),
    name='Private Insurance Average Premium'
))
fig.add_trace(go.Scatter(
    x=average_premium_commercial.index,
    y=average_premium_commercial.values,
    mode='lines',
    line=dict(color='green', dash='dot'),
    name='Commercial Insurance Average Premium'
))

fig.update_layout(
    title='Average Premium vs Production Year',
    xaxis_title='Production Year',
    yaxis_title='Average Premium Amount',
    plot_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(showgrid=True, gridcolor='lightgray'),
    yaxis=dict(showgrid=True, gridcolor='lightgray'),
    height=600,
    width=850,
    legend=dict(
        title='Legend',
        x=0.02,  # Positioning of the legend
        y=0.98,
        bgcolor='rgba(255,255,255,0.5)',
        bordercolor='black',
        borderwidth=1
    )
)
fig.show()

### Random Forest - Baisc Model

In [ ]:
###############################################################################
# Random Forest - Without Hyperparameter Tuning
###############################################################################
import time
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load dataset
forest_df = pd.read_csv('motor_data14-2018.csv')
# Select relevant columns and rename them for clarity
selected_columns = {
    "SEX": "sex",
    "SEATS_NUM": "seats_num",
    "CCM_TON": "ccm_ton",
    "INSR_TYPE": "insurance_type",
    "INSURED_VALUE": "insured_value",
    "PREMIUM": "premium",
    "PROD_YEAR": "production_year",
    "TYPE_VEHICLE": "vehicle_type",
    "USAGE": "usage"
}
forest_df = forest_df[list(selected_columns.keys())].rename(columns=selected_columns)

# Drop duplicates and null values for premium and production_year
forest_df.drop_duplicates(inplace=True)
forest_df.dropna(subset=["premium", "production_year"], inplace=True)
# Ensure premium is numeric and greater than 0
forest_df = forest_df[pd.to_numeric(forest_df['premium'], errors='coerce').gt(0)]
forest_df["insured_value"] = pd.to_numeric(forest_df["insured_value"], errors="coerce")
forest_df = forest_df[forest_df["insured_value"].gt(0)]  # Keep only positive values
forest_df = forest_df.sample(frac=0.1, random_state=42)
y = forest_df["premium"]
# Apply one-hot encoding to categorical variables
forest_df = pd.get_dummies(forest_df, columns=["sex", "insurance_type", "vehicle_type", "usage"], drop_first=True)
# Drop only the premium column, Train-test split
X = forest_df.drop(columns=["premium"])  
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train Random Forest
rf_model = RandomForestRegressor(max_depth=None, random_state=42)
rf_model.fit(X_train, y_train)

# Predictions
y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)
# Performance metrics
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

# Print results
print("\nRandom Forest Model Basic Performance:")
print("Train RMSE:", round(train_rmse, 2))
print("Test RMSE:", round(test_rmse, 2))
print("Train MAE:", round(train_mae, 2))
print("Test MAE:", round(test_mae, 2))
print("Train R²:", round(train_r2, 3))
print("Test R²:", round(test_r2, 3))

# Feature importance
feature_importance = rf_model.feature_importances_
feature_names = X.columns

#
importance_df_basic = pd.DataFrame({"Feature": feature_names, "Importance": feature_importance})
importance_df_basic = importance_df_basic.sort_values(by="Importance", ascending=False)


In [ ]:
###############################################################################
# Feature Importance
###############################################################################

import plotly.express as px

#
feature_labels = {
    "insured_value": "Insured Value",
    "production_year": "Production Year",
    "ccm_ton": "Engine Capacity (Ton)",
    "seats_num": "Number of Seats",
    "usage_Fare Paying Passengers": "Vehicle Usage"
}

# Top 5 features
top_5_features = importance_df_basic.head(5).copy()
top_5_features.loc[:, "Feature"] = top_5_features["Feature"].apply(lambda x: feature_labels.get(x, x))
print("\nTop 5 Most Important Features:")
print(top_5_features)

# Plot Top 5 Feature
fig = px.bar(
    top_5_features,
    x="Importance",
    y="Feature",
    orientation="h",
    title="Top 5 Most Important Features in Random Forest Model",
    labels={"Importance": "Importance Score", "Feature": "Feature"},
    text=top_5_features["Importance"].round(2),
)

fig.update_layout(
    yaxis=dict(categoryorder="total ascending"),
    xaxis_title="Importance Score",
    yaxis_title="Feature",
    template="plotly_white",
)

fig.show()

### Random Forest - Optimized with Hyperparameter Tunning

In [ ]:
###############################################################################
# Random Forest - With Hyperparameter Tuning
###############################################################################

import time
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load dataset
forest_df = pd.read_csv('motor_data14-2018.csv')
# Select relevant columns and rename them for clarity
selected_columns = {
    "SEX": "sex",
    "SEATS_NUM": "seats_num",
    "CCM_TON": "ccm_ton",
    "INSR_TYPE": "insurance_type",
    "INSURED_VALUE": "insured_value",
    "PREMIUM": "premium",
    "PROD_YEAR": "production_year",
    "TYPE_VEHICLE": "vehicle_type",
    "USAGE": "usage"
}
forest_df = forest_df[list(selected_columns.keys())].rename(columns=selected_columns)

# Drop duplicates and null values
forest_df.drop_duplicates(inplace=True)
forest_df.dropna(subset=["premium", "production_year"], inplace=True)

# Ensure premium is numeric and greater than 0
forest_df = forest_df[pd.to_numeric(forest_df['premium'], errors='coerce').gt(0)]
forest_df["insured_value"] = pd.to_numeric(forest_df["insured_value"], errors="coerce")
forest_df = forest_df[forest_df["insured_value"].gt(0)]  # Keep only positive values
forest_df = forest_df.sample(frac=0.1, random_state=42)
y = forest_df["premium"]
# Apply one-hot encoding to categorical variables
forest_df = pd.get_dummies(forest_df, columns=["sex", "insurance_type", "vehicle_type", "usage"], drop_first=True)

# Define features
X = forest_df.drop(columns=["premium"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Hyperparameter tuning setup
param_dist = {
    "n_estimators": [200, 300, 400],
    "max_depth": [None, 15, 20],
    "min_samples_split": [10, 20],
    "min_samples_leaf": [3, 5, 10],
    "max_features": ["sqrt", 0.3, 0.5],
    "max_samples": [0.6, 0.8, None]
}

# Initialize Random Forest
rf = RandomForestRegressor(random_state=42, verbose=0)

# Perform Randomized Search on hyperparameter
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=30,
    cv=3,
    scoring='neg_mean_squared_error',
    random_state=42
)

# Train using randomized search
random_search.fit(X_train, y_train)
# Get the best model
best_rf = random_search.best_estimator_
print("\nBest Parameters:", random_search.best_params_)

# Retrain with best parameters, increase no. of estimator
best_rf.fit(X_train, y_train)

# Predictions
y_train_pred = best_rf.predict(X_train)
y_test_pred = best_rf.predict(X_test)
# Performance metrics
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

#
print("\nTuned Model Performance:")
print("Train RMSE:", round(train_rmse, 2))
print("Test RMSE:", round(test_rmse, 2))
print("Train MAE:", round(train_mae, 2))
print("Test MAE:", round(test_mae, 2))
print("Train R²:", round(train_r2, 3))
print("Test R²:", round(test_r2, 3))

# Feature importance
feature_importance = best_rf.feature_importances_
feature_names = X.columns

#
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": feature_importance})
importance_df = importance_df.sort_values(by="Importance", ascending=False)

In [ ]:
import plotly.express as px

feature_labels = {
    "insured_value": "Insured Value",
    "production_year": "Production Year",
    "ccm_ton": "Engine Capacity (Ton)",
    "seats_num": "Number of Seats",
    "usage_Fare Paying Passengers": "Vehicle Usage"
}

# Top 5 most important features
top_5_features = importance_df.head(5).copy()
top_5_features.loc[:, "Feature"] = top_5_features["Feature"].apply(lambda x: feature_labels.get(x, x))
print("\nTop 5 Most Important Features:")
print(top_5_features)

#
fig = px.bar(
    top_5_features,
    x="Importance",
    y="Feature",
    orientation="h",
    title="Top 5 Most Important Features in Random Forest Model",
    labels={"Importance": "Importance Score", "Feature": "Feature"},
    text=top_5_features["Importance"].round(2),
)

fig.update_layout(
    yaxis=dict(categoryorder="total ascending"),
    xaxis_title="Importance Score",
    yaxis_title="Feature",
    template="plotly_white",
)

fig.show()

### Multiple Linear Regression

In [ ]:
# Import libraries
import pandas as pd
import numpy as np

In [ ]:
# Load datasets
vehicle_insurance_df = pd.read_csv('motor_data14-2018.csv')

# Select relevant columns and rename them for clarity
selected_columns = {
    "SEX": "sex",
    "INSR_BEGIN": "insurance_begin",
    "INSR_END": "insurance_end",
    "EFFECTIVE_YR": "effective_year",
    "INSR_TYPE": "insurance_type",
    "INSURED_VALUE": "insured_value",
    "PREMIUM": "premium",
    "OBJECT_ID": "object_id",
    "PROD_YEAR": "production_year",
    "SEATS_NUM": "seats_number",
    "CARRYING_CAPACITY": "carrying_capacity",
    "TYPE_VEHICLE": "vehicle_type",
    "CCM_TON": "ccm_ton",
    "MAKE": "manufacturer",
    "USAGE": "usage",
    "CLAIM_PAID": "claim_paid",
}

vehicle_insurance_df = vehicle_insurance_df[list(selected_columns.keys())].rename(columns=selected_columns)

# Replace sex values
vehicle_insurance_df["sex"] = vehicle_insurance_df["sex"].replace({0: "legal entity", 1: "male", 2: "female"})

# Replace insurance_type values
vehicle_insurance_df["insurance_type"] = vehicle_insurance_df["insurance_type"].replace({1201: "private", 1202: "commercial", 1204: "motor trade road risk"})

# Display data info
print(vehicle_insurance_df.info())
print(vehicle_insurance_df.head())

In [ ]:
# Drop duplicates
vehicle_insurance_df.drop_duplicates(inplace=True)

# Check for missing values
missing_values = vehicle_insurance_df.isnull().sum()
print("Missing values in each column:")
print(missing_values)

# Drop null values for premium, production_year, and manufacturer
vehicle_insurance_df.dropna(subset=["premium", "production_year", "manufacturer"], inplace=True)

# Convert production_year to an integer type
vehicle_insurance_df["production_year"] = vehicle_insurance_df["production_year"].astype(int)

# Check for missing values after dropping
missing_values_after = vehicle_insurance_df.isnull().sum()
print("Missing values after dropping:")
print(missing_values_after)
vehicle_insurance_df.dropna(subset=["premium", "production_year", "manufacturer"], inplace=True)

# Convert categorical variables to proper types
vehicle_insurance_df["sex"] = vehicle_insurance_df["sex"].astype("category")
vehicle_insurance_df["insurance_type"] = vehicle_insurance_df["insurance_type"].astype("category")
vehicle_insurance_df["vehicle_type"] = vehicle_insurance_df["vehicle_type"].astype("category")
vehicle_insurance_df["manufacturer"] = vehicle_insurance_df["manufacturer"].astype("category")
vehicle_insurance_df["usage"] = vehicle_insurance_df["usage"].astype("category")

# Check for zero values in each selected column
zero_values = (vehicle_insurance_df == 0).sum()
print("Zero values in each column:")
print(zero_values)

# Drop zero values in the premium column
vehicle_insurance_df = vehicle_insurance_df[vehicle_insurance_df["premium"] > 0]
vehicle_insurance_df.dropna(subset=["premium", "production_year", "manufacturer"], inplace=True)

# Replace 0s with median in the insured value column
median_insured_value_df = vehicle_insurance_df[vehicle_insurance_df['insured_value'] > 0].groupby(['manufacturer', 'vehicle_type', 'production_year'],
                                                                                                  dropna=False)['insured_value'].median().dropna().reset_index()
median_insured_value_dict = median_insured_value_df.set_index(['manufacturer', 'vehicle_type', 'production_year'])['insured_value'].to_dict()

def find_closest_year_match(manufacturer, vehicle_type, production_year):
    """Find the closest production year match if an exact match is not available, within a maximum difference of 5 years."""
    subset = median_insured_value_df[(median_insured_value_df['manufacturer'] == manufacturer) &
                                     (median_insured_value_df['vehicle_type'] == vehicle_type)]
    if subset.empty:
        return np.nan
    subset = subset[(subset['production_year'] - production_year).abs() <= 5]
    if subset.empty:
        return np.nan
    closest_year = subset.loc[(subset['production_year'] - production_year).abs().idxmin()]
    return closest_year['insured_value'] if not closest_year.empty else np.nan

def replace_zero_with_median(row):
    """Replace zero insured_value with the median value based on relaxed conditions."""
    key = (row['manufacturer'], row['vehicle_type'], row['production_year'])
    median_value = median_insured_value_dict.get(key, np.nan)
    
    if pd.notna(median_value) and median_value > 0:
        return median_value
    else:
        return find_closest_year_match(row['manufacturer'], row['vehicle_type'], row['production_year'])

# Apply replacements efficiently using vectorized approach
mask = vehicle_insurance_df['insured_value'] == 0
vehicle_insurance_df.loc[mask, 'insured_value'] = vehicle_insurance_df.loc[mask].apply(replace_zero_with_median, axis=1)

# Check for zero values after replacing
zero_values = (vehicle_insurance_df == 0).sum()
print("Zero values in each column after replacing:")
print(zero_values)

# Drop remaining zero-value rows in insured_value
vehicle_insurance_df = vehicle_insurance_df[vehicle_insurance_df['insured_value'] > 0]

# Check for zero values after dropping
zero_values = (vehicle_insurance_df == 0).sum()
print("Zero values in each column after dropping:")
print(zero_values)

In [ ]:
# Convert insurance_end and insurance_begin to date format
vehicle_insurance_df["insurance_begin"] = pd.to_datetime(vehicle_insurance_df["insurance_begin"], errors='coerce')
vehicle_insurance_df["insurance_end"] = pd.to_datetime(vehicle_insurance_df["insurance_end"], errors='coerce')

# Calculate insurance_duration in days
vehicle_insurance_df["insurance_duration"] = (vehicle_insurance_df["insurance_end"] - vehicle_insurance_df["insurance_begin"]).dt.days

# Create insurance_claim column
vehicle_insurance_df["insurance_claim"] = vehicle_insurance_df["claim_paid"].apply(lambda x: "claimed" if pd.notna(x) else "no claim")

# Take log of insured_value and premium
vehicle_insurance_df["log_insured_value"] = vehicle_insurance_df["insured_value"].apply(lambda x: np.log(x) if x > 0 else 0)
vehicle_insurance_df["log_premium"] = vehicle_insurance_df["premium"].apply(lambda x: np.log(x) if x > 0 else 0)

# Display cleaned data info
print(vehicle_insurance_df.info())
print(vehicle_insurance_df.head())

# Save the updated DataFrame (optional)
# vehicle_insurance_df.to_csv("/mnt/data/updated_vehicle_insurance.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Plot frequency plots for categorical variables
categorical_columns = ["sex", "insurance_type","vehicle_type", "usage"]

plt.figure(figsize=(15, 10))
for i, col in enumerate(categorical_columns, 1):
    plt.subplot(3, 2, i)
    sns.countplot(y=vehicle_insurance_df[col], order=vehicle_insurance_df[col].value_counts().index)
    plt.title(f"Frequency of {col}")
plt.tight_layout()
plt.show()

# Plot histograms for numerical variables
numerical_columns = ["production_year"]

plt.figure(figsize=(12, 8))
for i, col in enumerate(numerical_columns, 1):
    plt.subplot(2, 2, i)
    sns.histplot(vehicle_insurance_df[col], bins=50, kde=True)
    plt.title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

# Check the distribution of insured_value and premium
plt.figure(figsize=(12, 5))
for i, col in enumerate(["log_insured_value", "log_premium"], 1):
    plt.subplot(1, 2, i)
    sns.histplot(vehicle_insurance_df[col], bins=50, kde=True)
    plt.title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split

In [ ]:
# Create a new dataframe mlr_df with selected columns
mlr_df = vehicle_insurance_df[[  
    "sex", "insurance_type", "log_insured_value", "log_premium", "production_year",  
    "vehicle_type", "manufacturer", "usage", "insurance_duration", "insurance_claim"  
]].copy()

# Display cleaned data info
print(mlr_df.info())
print(mlr_df.head())

# filtered mlr_df to include only rows where insurance_duration > 300
mlr_df = mlr_df[mlr_df['insurance_duration'] > 300]

# # Display cleaned data info
# print(mlr_df.info())
# print(mlr_df.head())

In [ ]:
# Plot distributions for each column in mlr_df
for column in mlr_df.columns:
    plt.figure(figsize=(8, 5))
    if mlr_df[column].dtype == 'category' or mlr_df[column].dtype == 'object':
        sns.countplot(x=mlr_df[column])
        plt.title(f"Frequency of {column}")
    else:
        sns.histplot(mlr_df[column], bins=50, kde=True)
        plt.title(f"Distribution of {column}")
    plt.xlabel(column)
    plt.ylabel("Count")
    plt.show()
# Scatter plot between log_insured_value and log_premium
plt.figure(figsize=(8, 5))
sns.scatterplot(x=mlr_df['log_insured_value'], y=mlr_df['log_premium'])
plt.xlabel("Log Insured Value")
plt.ylabel("Log Premium")
plt.title("Scatter Plot between Log Insured Value and Log Premium")
plt.show()

In [ ]:
# Choose predictors (xi)
predictors = ['sex', 'insurance_type', 'log_insured_value', 'production_year', 'vehicle_type', 'usage', 'insurance_claim']

# Set predictors and target variable
X = mlr_df[predictors]
y = mlr_df['log_premium']

# Convert categorical variables into dummy variables
X = pd.get_dummies(X, drop_first=True).astype(float)

# Partition data into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Ensure numerical consistency
X_train = X_train.astype(float)
X_test = X_test.astype(float)

# Ensure alignment between X_train and y_train
X_train, y_train = X_train.align(y_train, join="inner", axis=0)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Stepwise selection procedure
def stepwise_selection(X, y, significance_level=0.05):
    included = list(X.columns)
    while True:
        model = sm.OLS(y, sm.add_constant(X[included])).fit()
        pvalues = model.pvalues.iloc[1:]
        max_pval = pvalues.max()
        if max_pval > significance_level:
            worst_feature = pvalues.idxmax()
            included.remove(worst_feature)
        else:
            break
    return included

selected_features = stepwise_selection(X_train, y_train)
print("Selected Features:", selected_features)

# Fit regression model with selected features
X_train_selected = sm.add_constant(X_train[selected_features])
X_test_selected = sm.add_constant(X_test[selected_features])
additive_model = sm.OLS(y_train, X_train_selected).fit()

# Get estimated model summary
print("Estimated Regression Model Summary:")
print(additive_model.summary())

# Predictions on both training and testing sets
y_train_pred = additive_model.predict(X_train_selected)
y_test_pred = additive_model.predict(X_test_selected)

# Residuals
train_residuals = additive_model.resid
test_residuals = y_test - y_test_pred

# Evaluate model performance on train set
print("MLE Regression Model Performance on Training Set:")
print(f"R² Score: {r2_score(np.exp(y_train), np.exp(y_train_pred)):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(np.exp(y_train), np.exp(y_train_pred))):.4f}")
print(f"MAE: {mean_absolute_error(np.exp(y_train), np.exp(y_train_pred)):.4f}\n")

# Evaluate model performance on test set
print("MLE Regression Model Performance on Testing Set:")
print(f"R² Score: {r2_score(np.exp(y_test), np.exp(y_test_pred)):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(np.exp(y_test), np.exp(y_test_pred))):.4f}")
print(f"MAE: {mean_absolute_error(np.exp(y_test), np.exp(y_test_pred)):.4f}\n")

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from statsmodels.stats.diagnostic import het_breuschpagan, het_white, linear_reset
from scipy.stats import kstest, skew, kurtosis
from statsmodels.stats.outliers_influence import variance_inflation_factor
    
# 1. Linearity: Ramsey RESET test
test_reset = linear_reset(additive_model, power=2, use_f=True)
print(f"Ramsey RESET Test p-value: {test_reset.pvalue:.4f}")

# 1. Linearity: Residuals vs Fitted Values Plot
plt.figure(figsize=(6,4))
sns.scatterplot(x=y_train_pred, y=train_residuals, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Linearity: Residuals vs Fitted Values")
plt.savefig("linearity_plot.png")
plt.show()

# 2. Normality of Residuals: Kolmogorov-Smirnov Test, Skewness, Kurtosis
ks_stat, ks_p_value = kstest(train_residuals, 'norm', args=(np.mean(train_residuals), np.std(train_residuals)))
print(f"Kolmogorov-Smirnov Test: statistic={ks_stat:.4f}, p-value={ks_p_value:.4f}")
print(f"Skewness: {skew(train_residuals):.4f}")
print(f"Kurtosis: {kurtosis(train_residuals):.4f}\n")

# 2. Normality: Q-Q Plot
sm.qqplot(train_residuals, line='45', fit=True)
plt.title("Q-Q Plot for Normality of Residuals")
plt.savefig("qq_plot.png")
plt.show()

# 3. Homoscedasticity: Scale-Location Plot
plt.figure(figsize=(6,4))
sns.scatterplot(x=y_train_pred, y=np.sqrt(abs(train_residuals)), alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("√|Residuals|")
plt.title("Scale-Location Plot for Homoscedasticity")
plt.savefig("scale_location_plot.png")
plt.show()

# 4. Multicollinearity: Variance Inflation Factor (VIF)
vif_data = pd.DataFrame()
vif_data["Variable"] = X_train_selected.columns
vif_data["VIF"] = [variance_inflation_factor(X_train_selected.values, i) for i in range(X_train_selected.shape[1])]
print("Variance Inflation Factor (VIF):")
print(vif_data)

**Linearity Issue (Ramsey RESET Test)**\
Interpretation:\
The model misses some nonlinear relationships.\
Fixes:
- Include interaction or polynomial terms (e.g., quadratic/cubic transformations of key predictors).
- Try feature engineering (log, square root, or power transformations).
- Consider a non-linear model (e.g., Generalized Additive Models (GAMs)) if transformations don't help.

**Normality Issue (Kolmogorov-Smirnov Test, Q-Q Plot)**\
Fixes:
- Apply log transformation to the dependent variable.
- Use robust regression (e.g., Huber Regression) if outliers are an issue.
- Check if removing influential points (Cook’s Distance > 1) improves normality.



In [ ]:
from statsmodels.stats.outliers_influence import OLSInfluence

# Influential Outliers: Cook’s Distance
influence = OLSInfluence(additive_model)
cooks_d, _ = influence.cooks_distance

# Identify influential outliers (threshold: 4/n)
influential_threshold = 4 / len(cooks_d)
influential_outliers = X_train_selected.index[np.where(cooks_d > influential_threshold)[0]]
# print("Influential Outliers (Indices):", influential_outliers.tolist())

plt.figure(figsize=(6,4))
plt.stem(range(len(cooks_d)), cooks_d, markerfmt=",")
plt.axhline(y=influential_threshold, color='r', linestyle='--', label="Threshold: 4/n")
plt.xlabel("Observation Index")
plt.ylabel("Cook’s Distance")
plt.title("Cook’s Distance for Influential Outliers")
plt.legend()
plt.savefig("cooks_distance.png")
plt.show()

# Dealing with Influential Outliers
# Remove outliers and refit the model
X_train_filtered = X_train_selected.drop(index=influential_outliers, errors='ignore')
y_train_filtered = y_train.drop(index=influential_outliers, errors='ignore')

# Ensure alignment
X_train_filtered, y_train_filtered = X_train_filtered.align(y_train_filtered, join="inner", axis=0)

# Fit the model after removing influential outliers, using robust standard errors
additive_model_filtered = sm.OLS(y_train_filtered, X_train_filtered).fit(cov_type='HC3')
print("Refitted Model Summary after Removing Influential Outliers:")
print(additive_model_filtered.summary())

# Predictions on both training and testing sets
y_train_filtered_pred = additive_model_filtered.predict(X_train_filtered)
y_test_filtered_pred = additive_model_filtered.predict(X_test_selected)

# Residuals
train_filtered_residuals = additive_model_filtered.resid
test_filtered_residuals = y_test - y_test_filtered_pred

# Evaluate model performance on train set
print("MLE Regression Model Performance after Removing Influential Outliers on Training Set:")
print(f"R² Score: {r2_score(np.exp(y_train_filtered), np.exp(y_train_filtered_pred)):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(np.exp(y_train_filtered), np.exp(y_train_filtered_pred))):.4f}")
print(f"MAE: {mean_absolute_error(np.exp(y_train_filtered), np.exp(y_train_filtered_pred)):.4f}\n")

# Evaluate model performance on test set
print("MLE Regression Model Performance after Removing Influential Outliers on Testing Set:")
print(f"R² Score: {r2_score(np.exp(y_test), np.exp(y_test_filtered_pred)):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(np.exp(y_test), np.exp(y_test_filtered_pred))):.4f}")
print(f"MAE: {mean_absolute_error(np.exp(y_test), np.exp(y_test_filtered_pred)):.4f}\n")

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from statsmodels.stats.diagnostic import het_breuschpagan, het_white, linear_reset
from scipy.stats import kstest, skew, kurtosis
from statsmodels.stats.outliers_influence import variance_inflation_factor
    
# 1. Linearity: Ramsey RESET test
test_reset = linear_reset(additive_model_filtered, power=2, use_f=True)
print(f"Ramsey RESET Test p-value: {test_reset.pvalue:.4f}")

# 1. Linearity: Residuals vs Fitted Values Plot
plt.figure(figsize=(6,4))
sns.scatterplot(x=y_train_filtered_pred, y=train_filtered_residuals, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Linearity: Residuals vs Fitted Values")
plt.savefig("linearity_plot.png")
plt.show()

# 2. Normality of Residuals: Kolmogorov-Smirnov Test, Skewness, Kurtosis
ks_stat, ks_p_value = kstest(train_filtered_residuals, 'norm', args=(np.mean(train_filtered_residuals), np.std(train_filtered_residuals)))
print(f"Kolmogorov-Smirnov Test: statistic={ks_stat:.4f}, p-value={ks_p_value:.4f}")
print(f"Skewness: {skew(train_filtered_residuals):.4f}")
print(f"Kurtosis: {kurtosis(train_filtered_residuals):.4f}\n")

# 2. Normality: Q-Q Plot
sm.qqplot(train_filtered_residuals, line='45', fit=True)
plt.title("Q-Q Plot for Normality of Residuals")
plt.savefig("qq_plot.png")
plt.show()

# 3. Homoscedasticity: Scale-Location Plot
plt.figure(figsize=(6,4))
sns.scatterplot(x=y_train_filtered_pred, y=np.sqrt(abs(train_filtered_residuals)), alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("√|Residuals|")
plt.title("Scale-Location Plot for Homoscedasticity")
plt.savefig("scale_location_plot.png")
plt.show()

# 4. Multicollinearity: Variance Inflation Factor (VIF)
vif_data = pd.DataFrame()
vif_data["Variable"] = X_train_filtered.columns
vif_data["VIF"] = [variance_inflation_factor(X_train_filtered.values, i) for i in range(X_train_filtered.shape[1])]
print("Variance Inflation Factor (VIF):")
print(vif_data)

In [ ]:
# Add interaction terms
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_train_interact = pd.DataFrame(poly.fit_transform(X_train_filtered), columns=poly.get_feature_names_out(X_train_filtered.columns), index=X_train_filtered.index)
X_test_interact = pd.DataFrame(poly.transform(X_test_selected), columns=poly.get_feature_names_out(X_test_selected.columns), index=X_test_selected.index)

# Fit the model after adding interaction terms with robust standard errors
interact_model = sm.OLS(y_train_filtered, X_train_interact).fit(cov_type='HC3')
print("Refitted Model Summary with Interaction Terms and Robust Standard Errors:")
print(interact_model.summary())

# Predictions
y_test_interact_pred = interact_model.predict(X_test_interact)
y_train_interact_pred = interact_model.predict(X_train_interact)

# Evaluate model performance
print("Interaction Model Performance on Training Set:")
print(f"R² Score: {r2_score(np.exp(y_train_filtered), np.exp(y_train_interact_pred)):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(np.exp(y_train_filtered), np.exp(y_train_interact_pred))):.4f}")
print(f"MAE: {mean_absolute_error(np.exp(y_train_filtered), np.exp(y_train_interact_pred)):.4f}\n")

print("Interaction Model Performance on Testing Set:")
print(f"R² Score: {r2_score(np.exp(y_test), np.exp(y_test_interact_pred)):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(np.exp(y_test), np.exp(y_test_interact_pred))):.4f}")
print(f"MAE: {mean_absolute_error(np.exp(y_test), np.exp(y_test_interact_pred)):.4f}\n")

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_filtered)
X_test_scaled = scaler.transform(X_test_selected)

# Add polynomial term for log_insured_value
X_train_poly = X_train_filtered.copy()
X_test_poly = X_test_selected.copy()
X_train_poly['log_insured_value^2'] = X_train_poly['log_insured_value'] ** 2
X_test_poly['log_insured_value^2'] = X_test_poly['log_insured_value'] ** 2

# Fit a regression model to training data
X_train_poly = sm.add_constant(X_train_poly)  # Add constant for intercept
poly_model = sm.OLS(y_train_filtered, X_train_poly).fit()

# Get estimated model summary
print("Estimated Polynomial Regression Model Summary:")
print(poly_model.summary())

# Predict on test set
X_test_poly = sm.add_constant(X_test_poly)  # Add constant
y_poly_pred = poly_model.predict(X_test_poly)

# Evaluate model performance on train set
y_train_poly_pred = poly_model.predict(X_train_poly)
print("Polynomial Regression Model Performance on Training Set:")
print(f"R² Score: {r2_score(np.exp(y_train_filtered), np.exp(y_train_poly_pred)):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(np.exp(y_train_filtered), np.exp(y_train_poly_pred))):.4f}")
print(f"MAE: {mean_absolute_error(np.exp(y_train_filtered), np.exp(y_train_poly_pred)):.4f}\n")

# Evaluate model performance on test set
print("Polynomial Regression Model Performance on Testing Set:")
print(f"R² Score: {r2_score(np.exp(y_test), np.exp(y_poly_pred)):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(np.exp(y_test), np.exp(y_poly_pred))):.4f}")
print(f"MAE: {mean_absolute_error(np.exp(y_test), np.exp(y_poly_pred)):.4f}\n")

#### Hyperparameter Tuning 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_filtered)
X_test_scaled = scaler.transform(X_test_selected)

# Lasso Regression with Grid Search
lasso = Lasso()
lasso_params = {'alpha': np.logspace(-2, 2, 5)}
lasso_cv = GridSearchCV(lasso, lasso_params, cv=3, scoring='r2')
lasso_cv.fit(X_train_scaled, y_train_filtered)
best_alpha_lasso = lasso_cv.best_params_['alpha']
lasso_opt = Lasso(alpha=best_alpha_lasso).fit(X_train_scaled, y_train_filtered)

# Ridge Regression with Grid Search
ridge = Ridge()
ridge_params = {'alpha': np.logspace(-2, 2, 5)}
ridge_cv = GridSearchCV(ridge, ridge_params, cv=3, scoring='r2')
ridge_cv.fit(X_train_scaled, y_train_filtered)
best_alpha_ridge = ridge_cv.best_params_['alpha']
ridge_opt = Ridge(alpha=best_alpha_ridge).fit(X_train_scaled, y_train_filtered)

# Elastic Net with Randomized Search
elastic_params = {
    'alpha': np.logspace(-2, 2, 5),
    'l1_ratio': np.linspace(0.1, 0.9, 5)
}
elastic_net_cv = RandomizedSearchCV(
    ElasticNet(),
    elastic_params,
    n_iter=10,
    cv=3,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)
elastic_net_cv.fit(X_train_scaled, y_train_filtered)
best_alpha_elastic = elastic_net_cv.best_params_['alpha']
best_l1_ratio_elastic = elastic_net_cv.best_params_['l1_ratio']
elastic_opt = ElasticNet(alpha=best_alpha_elastic, l1_ratio=best_l1_ratio_elastic).fit(X_train_scaled, y_train_filtered)

# Predictions and Performance Evaluation
models = {
    "Lasso": lasso_opt,
    "Ridge": ridge_opt,
    "Elastic Net": elastic_opt
}

results = []
for name, model in models.items():
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)

    results.append({
        "Model": name,
        "Best Alpha": model.alpha if hasattr(model, "alpha") else "N/A",
        "Train R²": r2_score(np.exp(y_train_filtered), np.exp(y_train_pred)),
        "Test R²": r2_score(np.exp(y_test), np.exp(y_test_pred)),
        "Train RMSE": np.sqrt(mean_squared_error(np.exp(y_train_filtered), np.exp(y_train_pred))),
        "Test RMSE": np.sqrt(mean_squared_error(np.exp(y_test), np.exp(y_test_pred))),
        "Train MAE": mean_absolute_error(np.exp(y_train_filtered), np.exp(y_train_pred)),
        "Test MAE": mean_absolute_error(np.exp(y_test), np.exp(y_test_pred)),
        "Selected Features (Non-Zero Coefficients)": np.sum(model.coef_ != 0) if hasattr(model, "coef_") else "All Features"
    })

# Convert results to DataFrame and display
results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
# Fit Ridge regression
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=best_alpha_ridge)
ridge.fit(X_train_scaled, y_train_filtered)

# Compute Adjusted VIF After Ridge
ridge_vif = pd.DataFrame()
ridge_vif["Feature"] = X_train_filtered.columns
ridge_vif["Adjusted VIF After Ridge"] = [1 / (1 - r2) if r2 < 1 else float('inf') 
                                         for r2 in ridge.coef_**2 / ridge.coef_.var()]

print(ridge_vif.sort_values(by="Adjusted VIF After Ridge", ascending=False))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

# Make a copy of the original dataset to preserve it
X_train_lasso = X_train_filtered.copy()

# Scale the features (important for Lasso)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_lasso)

# Train Lasso with Cross-Validation
lasso = LassoCV(alphas=np.logspace(-3, 3, 100), cv=5, random_state=42, max_iter=5000)
lasso.fit(X_train_scaled, y_train_filtered)

# Identify selected (non-zero) features
selected_features = X_train_lasso.columns[lasso.coef_ != 0]

# Filter dataset to keep only selected features
X_train_lasso_selected = X_train_lasso[selected_features]

# Compute VIF for selected features
vif_data = pd.DataFrame()
vif_data["Feature"] = X_train_lasso_selected.columns
vif_data["VIF After Lasso"] = [variance_inflation_factor(X_train_lasso_selected.values, i) for i in range(X_train_lasso_selected.shape[1])]

# Display results
print(vif_data.sort_values(by="VIF After Lasso", ascending=False))